# 第8回：クラスを予測する—分類

**今日の問い：正解率だけで十分なのはどんなときか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
model = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]


## TRY：閾値0.5で混同行列を読む


In [ ]:
prediction = (probability >= 0.5).astype(int)
print("accuracy:", round(accuracy_score(y_valid, prediction), 3))
print("precision:", round(precision_score(y_valid, prediction), 3))
print("recall:", round(recall_score(y_valid, prediction), 3))
print("F1:", round(f1_score(y_valid, prediction), 3))
ConfusionMatrixDisplay.from_predictions(y_valid, prediction, display_labels=["非活性", "活性"], cmap="Blues")
plt.title("混同行列")


## TRY：判定閾値を変える


In [ ]:
rows=[]
for threshold in [0.3, 0.5, 0.7]:
    pred=(probability >= threshold).astype(int)
    rows.append({"閾値": threshold, "precision": precision_score(y_valid, pred), "recall": recall_score(y_valid, pred), "F1": f1_score(y_valid, pred)})
pd.DataFrame(rows).round(3)


## 話し合い

活性候補を見逃したくない探索段階ならrecall、追試コストが非常に高い絞り込み段階ならprecisionを重く見る、といった使い分けが考えられます。正解は利用場面で変わります。
